# HyperKT Objective 01 — Stage 11: Publication-Ready Analysis

**Purpose:** audit Stage 10 frozen test outputs, generate manuscript-ready tables/figures, and draft evidence-constrained Results, Discussion, and Limitations. **No retraining, no new model selection, no test-set tuning.**

**Input:** `hyperkt_stage10.zip` (upload when prompted). **Output:** `hyperkt_stage11.zip`.

**Interpretation:** All comparisons are observational post-hoc summaries of the pre-registered frozen models. DKT is an independently implemented baseline; fixed Stage 3B hypergraph across seeds. Student-cluster bootstrap intervals are conditional on the trained models.

In [ ]:
import io, json, zipfile, hashlib, os, pathlib, sys
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss, f1_score, accuracy_score
from IPython.display import display
OUT=pathlib.Path('/content/hyperkt_stage11') if pathlib.Path('/content').exists() else pathlib.Path('hyperkt_stage11')
OUT.mkdir(exist_ok=True,parents=True)
try:
 from google.colab import files
 uploaded=files.upload()
 path=next((pathlib.Path(k) for k in uploaded if 'stage10' in k.lower() and k.lower().endswith('.zip')),None)
 if path is None: raise ValueError('Upload hyperkt_stage10.zip')
except ImportError:
 path=pathlib.Path('/mnt/data/hyperkt_stage10.zip')
 if not path.exists(): path=pathlib.Path('hyperkt_stage10.zip')
assert path.exists(),f'Not found: {path}'
z=zipfile.ZipFile(path)
print('Input:',path,'SHA256:',hashlib.sha256(path.read_bytes()).hexdigest())
print('Members:',len(z.namelist()))

## 1. Recompute primary summary and verify file integrity

In [ ]:
required=['per_seed_test_metrics.csv','five_seed_test_summary.csv','evaluation_protocol.json','paired_auc_comparisons.csv']
assert all(k in z.namelist() for k in required)
metrics=pd.read_csv(io.BytesIO(z.read('per_seed_test_metrics.csv')))
protocol=json.loads(z.read('evaluation_protocol.json'))
print('Protocol:',protocol['stage'], 'One-time held-out:',protocol['one_time_held_out_test'])
print('Metric columns:',list(metrics.columns))
assert protocol['checkpoint_count']==15
assert len(metrics)==15
assert set(metrics['variant'])=={'DKT_matched','HyperKT_matched','HyperKT_gated'}
assert all(metrics.groupby('variant').size()==5)
assert metrics[['auc','nll','brier','f1']].notna().all().all()
summary=metrics.groupby('variant')[['auc','nll','brier','f1','ece15','accuracy','precision','recall']].agg(['mean','std'])
summary.to_csv(OUT/'table_primary_metrics.csv')
display(summary.round(6))

## 2. Audit saved predictions and recompute per-seed AUC

This checks that all models use the same outcome labels and student identifiers in the same order. It does not independently reconstruct the Stage 1 event identifiers.

In [ ]:
predfiles=sorted(k for k in z.namelist() if k.startswith('predictions_') and k.endswith('.npz'))
assert len(predfiles)==15
for k in predfiles[:1]:
 with np.load(io.BytesIO(z.read(k))) as a:
  print(k,{q:(a[q].shape,str(a[q].dtype)) for q in a.files})


In [ ]:
arrays={}
for k in predfiles:
 with np.load(io.BytesIO(z.read(k))) as f:
  arrays[k]={name:f[name].copy() for name in f.files}
keys=set.intersection(*(set(x) for x in arrays.values()))
print('Common keys:',keys)
# Accept explicit, documented aliases only
labelkey=next((q for q in ['y','labels','y_true','target','targets'] if q in keys),None)
probkey=next((q for q in ['p','probs','y_prob','pred','predictions'] if q in keys),None)
studentkey=next((q for q in ['student','students','student_ids','user','users','user_id'] if q in keys),None)
assert labelkey and probkey and studentkey, 'Inspect NPZ key names and explicitly update key mappings; do not guess.'
ref=arrays[predfiles[0]]
checks=[]
for k,a in arrays.items():
 y=np.asarray(a[labelkey]).ravel(); p=np.asarray(a[probkey]).ravel(); ids=np.asarray(a[studentkey]).ravel()
 assert len(y)==len(p)==len(ids)==32522
 assert np.array_equal(y,np.asarray(ref[labelkey]).ravel()),'Label ordering mismatch'
 assert np.array_equal(ids,np.asarray(ref[studentkey]).ravel()),'Student ordering mismatch'
 assert np.isfinite(p).all() and ((p>=0)&(p<=1)).all()
 modelseed=k.removeprefix('predictions_').removesuffix('.npz')
 seed=int(modelseed.rsplit('_seed',1)[1]); variant=modelseed.rsplit('_seed',1)[0]
 row=metrics[(metrics.variant==variant)&(metrics.seed==seed)].iloc[0]
 auc=roc_auc_score(y,p)
 checks.append(dict(variant=variant,seed=seed,recomputed_auc=auc,saved_auc=row.auc,absolute_error=abs(auc-row.auc),n=len(y),students=len(np.unique(ids))))
audit=pd.DataFrame(checks)
assert audit.absolute_error.max()<1e-6
assert (audit.students==402).all()
audit.to_csv(OUT/'prediction_alignment_audit.csv',index=False)
display(audit.round(8))

## 3. Descriptive comparisons and paper figures

The 5 training seeds share one test cohort; they are not five independent test datasets. Do not treat seed-level standard deviation as population uncertainty.

In [ ]:
means=metrics.groupby('variant')[['auc','nll','brier','f1','ece15']].mean()
order=['DKT_matched','HyperKT_matched','HyperKT_gated']
labels=['DKT (independent)','HyperKT (original)','HyperKT (gated)']
for metric,title in [('auc','Held-out ROC-AUC'),('f1','Held-out F1'),('nll','Held-out negative log-likelihood'),('brier','Held-out Brier score')]:
 fig,ax=plt.subplots(figsize=(7.5,4.2))
 v=means.loc[order,metric].values
 ax.bar(labels,v)
 ax.set_ylabel(metric.upper());ax.set_title(title)
 ax.tick_params(axis='x',rotation=12)
 for i,x in enumerate(v):ax.text(i,x,f'{x:.4f}',ha='center',va='bottom',fontsize=9)
 fig.tight_layout();fig.savefig(OUT/f'figure_{metric}.png',dpi=220,bbox_inches='tight');plt.close(fig)
print('Figures generated')

## 4. Existing student-clustered paired AUC intervals

Use the intervals produced during the one-time Stage 10 evaluation; do not run additional test-based model selection.

In [ ]:
paired=pd.read_csv(io.BytesIO(z.read('paired_auc_comparisons.csv')))
paired.to_csv(OUT/'table_paired_auc.csv',index=False)
cluster=pd.read_csv(io.BytesIO(z.read('student_cluster_auc_intervals.csv')))
cluster.to_csv(OUT/'table_clustered_auc_intervals.csv',index=False)
display(paired)
print('DKT minus original HyperKT mean:',paired.loc[paired.comparator=='HyperKT_matched','dkt_minus_comparator_auc'].mean())

## 5. Draft publication text with limitations

In [ ]:
s=summary
fmt=lambda model,metric: f"{s.loc[model,(metric,'mean')]:.4f} ± {s.loc[model,(metric,'std')]:.4f}"
paragraph=f"""# Results and Discussion — evidence-constrained draft

## Experimental protocol

The final evaluation used the ASSISTments 2009 student-disjoint held-out split comprising 402 students and 32,522 prediction targets per checkpoint. Five previously frozen checkpoints were evaluated for each of three configurations: independently implemented DKT, original HyperKT, and gated HyperKT. Model selection and early stopping were conducted using validation data. The held-out test split was evaluated without retraining. The Stage 9E experiment used batch size 32, learning rate 0.0002, maximum 20 epochs, and patience 5. Hypergraph embeddings were fixed across the five training seeds.

## Predictive discrimination

DKT achieved test ROC-AUC {fmt('DKT_matched','auc')}, compared with {fmt('HyperKT_matched','auc')} for original HyperKT and {fmt('HyperKT_gated','auc')} for gated HyperKT (mean ± standard deviation over five training seeds). The DKT–original HyperKT mean AUC difference was {means.loc['DKT_matched','auc']-means.loc['HyperKT_matched','auc']:.4f}. Original HyperKT had mean F1 {means.loc['HyperKT_matched','f1']:.4f}, compared with {means.loc['DKT_matched','f1']:.4f} for DKT. These metrics capture different properties, and the F1 difference does not establish overall superiority.

## Calibration and error

DKT achieved mean test negative log-likelihood {means.loc['DKT_matched','nll']:.4f} and Brier score {means.loc['DKT_matched','brier']:.4f}. Original HyperKT achieved {means.loc['HyperKT_matched','nll']:.4f} and {means.loc['HyperKT_matched','brier']:.4f}, respectively. The observed differences are small; uncertainty calibration improvement is not established by these results.

## Interpretation and limitations

The original HyperKT did not outperform the independent DKT baseline in test ROC-AUC. Gated hypergraph fusion did not improve the original architecture. The DKT implementation was not certified as equivalent to the published reference implementation, and the architectures differ in parameter count. The Stage 3B hypergraph was reused across seeds, so seed-level variation excludes uncertainty in hypergraph construction. The positional decay models interaction distance, not actual elapsed time. Student-clustered bootstrap intervals conditional on individual checkpoints should not be interpreted as confidence intervals over independent datasets. Event uniqueness passed the earlier Stage 9F audit, but was not independently reconstructed during Stage 10. All findings concern one dataset and one student-disjoint split. No causal-prerequisite discovery claim is justified by the co-occurrence graph.

## Reproducibility

The evaluation archive contains checkpoint hashes, a frozen evaluation protocol, per-seed test metrics, per-student predictions, and paired AUC comparisons. This analysis reproduces summary statistics and audits prediction alignment. No additional test-set optimization was performed.

**Editorial note:** This is a draft based on recorded results. Verify baseline architecture against primary sources and add verified citations before submission. Do not describe HyperKT as state of the art or superior to DKT.
"""
(OUT/'manuscript_results_discussion.md').write_text(paragraph)
print(paragraph[:2100])

## 6. Reproducibility manifest and export

In [ ]:
manifest={'stage':'11','source_archive_sha256':hashlib.sha256(path.read_bytes()).hexdigest(),'source_protocol':protocol,'models':order,'test_set_reused_for_tuning':False,'analysis_type':'descriptive post-hoc reporting; no training','limitations':['independent DKT baseline fidelity','fixed pretrained hypergraph across seeds','single dataset/split','test event uniqueness not independently reconstructed in Stage 10']}
(OUT/'analysis_manifest.json').write_text(json.dumps(manifest,indent=2))
zip_path=OUT.parent/'hyperkt_stage11.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as archive:
 for f in sorted(OUT.iterdir()):archive.write(f,arcname=f.name)
print('Created',zip_path,'size:',zip_path.stat().st_size)
try:
 from google.colab import files
 files.download(str(zip_path))
except ImportError:pass